# Arm D: the natural merge (DDR + all EyePACS, no source balancing) on the group-aware DDR split
**Why:** arm C (the old merge) keeps only EyePACS grades 1, 3 and 4, so the dataset name predicts disease almost by construction. Arm D is the realistic merge: **all** EyePACS grades added in their natural mix, with **no** source balancing. Comparing **D with B** (same data, B adds source-balancing) shows whether the gain on unseen sources comes from the balancing sampler, and comparing **D with A** shows whether a realistic merge helps at all.

Trains arm D for seeds 42, 43, 44 on the same group-aware DDR split as the other arms (EyePACS and APTOS splits unchanged, mild augmentation, 10 epochs), then evaluates it on the same test sets and compares it with arms A, B and C. About **4 hours** on a GPU (about 75 minutes per seed, like arm B); finished runs are skipped if you rerun, so it can be done over several sessions.

**Before running**
* Accelerator **GPU**, Internet **On** (ImageNet weights).
* Add as inputs: (1) the output of your original run (processed images, `manifest.csv`), (2) the output of **`ddr_group_split`** (`audit/ddr_splits_all.csv`), (3) the output of **`evaluate_on_group_split`** (`v2/runs/arm_*/seed*/results.json`, for the comparison with A, B and C). No code dataset is needed.
* After training, **Save Version -> Save & Run All (Commit)** so the results are kept. The evaluation part has a declaration gate (it opens the test sets again).

**What to send me:** the final comparison tables.

In [ ]:
import base64, glob, json, os, shutil, subprocess, sys, zlib
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("EARLYDR_WORK", "/kaggle/working")
CODE = os.path.join(WORK, "code")
EXTRA = os.environ.get("EARLYDR_TEST", "").split()     # test hook, leave empty on Kaggle
CODE_FILES = (
    "..."
    "/wcl8P0K"
)
try:
    import timm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"])
for name, text in json.loads(zlib.decompress(base64.b64decode(CODE_FILES))).items():
    path = os.path.join(CODE, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(text)
print("code written to", CODE)
sys.path.insert(0, CODE)

def real_dirs(pattern):
    return [os.path.dirname(f) for f in glob.glob(os.path.join(INPUT, "**", pattern), recursive=True)]

def processed_in(folder):
    return [p for p in glob.glob(os.path.join(folder, "processed_*")) if os.path.isdir(p) and not os.path.islink(p)]

# build the v2 work folder: original manifest + the group split for DDR + the original processed images
data_dirs = [d for d in real_dirs("manifest.csv") if processed_in(d)]
split_files = glob.glob(os.path.join(INPUT, "**", "audit", "ddr_splits_all.csv"), recursive=True)
if not data_dirs:
    sys.exit("STOP: add the output of your original run (it has manifest.csv and the processed images) as an input.")
if not split_files:
    sys.exit("STOP: add the output of the ddr_group_split notebook (it has audit/ddr_splits_all.csv) as an input.")
DATA, SPLITS = data_dirs[0], split_files[0]
print("images and manifest from:", DATA, "\nsplits from:", SPLITS)

import pandas as pd
V2 = os.path.join(WORK, "v2")
os.makedirs(V2, exist_ok=True)
manifest = pd.read_csv(os.path.join(DATA, "manifest.csv"))
groups = pd.read_csv(SPLITS).set_index("image_id")["group"]
is_ddr = manifest.source == "DDR"
assert manifest[is_ddr].image_id.isin(groups.index).all(), "split file does not cover every DDR image"
manifest.loc[is_ddr, "split"] = manifest.loc[is_ddr, "image_id"].map(groups).values
manifest.to_csv(os.path.join(V2, "manifest.csv"), index=False)
manifest.to_csv(os.path.join(V2, "splits.csv"), index=False)
processed = processed_in(DATA)[0]
link = os.path.join(V2, os.path.basename(processed))
if not os.path.exists(link):
    os.symlink(processed, link)
print(manifest.groupby(["source", "split"]).size().unstack(fill_value=0))

## Train arm D (3 seeds)

In [ ]:
SEEDS = [42, 43, 44]
EPOCHS = 10
subprocess.run([sys.executable, "-m", "early_dr.run_experiment", "configs/arms/arm_D.yaml", "--set",
                "task=stage1", f"paths.work={V2}", f"train.epochs={EPOCHS}", *EXTRA,
                "--stage", "train", "--seeds", *map(str, SEEDS)], cwd=CODE)
rows = []
for f in sorted(glob.glob(os.path.join(V2, "runs", "arm_D", "seed*", "train_info.json"))):
    info = json.load(open(f))
    rows.append({"run": "arm_D", "seed": f.split(os.sep)[-2], "best validation PR-AUC": round(info["best_val_score"], 4), "epochs": len(info["history"])})
print(pd.DataFrame(rows).to_string(index=False))
print("\nThe validation set of arm D is DDR + EyePACS, the same as arm B, so these two scores are comparable (B: about 0.897).")

## Declare this analysis, then confirm

In [ ]:
import datetime
TEXT = f"""# Declaration for the arm D analysis
Date declared: {datetime.date.today()}   (written BEFORE the test sets are opened for arm D)

This is a new analysis, added after the original declared analysis and after seeing its results.
* Model: arm D (DDR + all EyePACS grades, natural mix, no source balancing), seeds 42, 43 and 44, on the group-aware DDR split.
* Test sets: DDR group-split test, held-out EyePACS patients, APTOS, and the arm's own pooled test split. Each is opened once for this analysis.
* Comparison: arms A, B and C on the same split (from evaluate_on_group_split). D vs B isolates the source-balancing sampler; D vs A tests whether a realistic merge helps.
* Metrics: AUROC, PR-AUC, sensitivity and specificity at the frozen threshold (0.90 specificity target), ECE, Brier, with patient-level bootstrap intervals. Per-grade and specificity metrics are post-hoc.
"""
CONFIRM_DECLARATION = False
if not CONFIRM_DECLARATION:
    print(TEXT)
    sys.exit("Read the declaration above, set CONFIRM_DECLARATION = True and run this cell again.")
open(os.path.join(V2, "DECLARATION_arm_D.md"), "w").write(TEXT)
print("declaration saved")

## Evaluate arm D (internal test sets, then APTOS)

In [ ]:
for flag in ([], ["--external"]):
    subprocess.run([sys.executable, "-m", "early_dr.run_experiment", "configs/arms/arm_D.yaml", "--set", "task=stage1",
                    f"paths.work={V2}", *EXTRA, "--stage", "evaluate", "--seeds", *map(str, SEEDS), *flag], cwd=CODE)
print(open(os.path.join(V2, "test_opened.log")).read())

## Compare arms A, B, C and D
Mean over seeds of the estimate and of the per-seed 95% patient-level bootstrap interval (early DR vs No DR). ECE and Brier cover any DR vs No DR on all grades. A, B and C come from the `evaluate_on_group_split` output; their own-test sets differ by arm, so `arm_test` rows are not comparable across arms.

In [ ]:
rows = []
def collect(pattern, tag):
    for f in glob.glob(pattern, recursive=True):
        seed_dir = os.path.dirname(f); arm = os.path.basename(os.path.dirname(seed_dir)); seed = int(os.path.basename(seed_dir)[4:])
        if tag == "others" and arm == "arm_D": continue
        res = json.load(open(f))
        for source, r in res.items():
            e = r.get("early_vs_none")
            if not e or source == "arm_test": continue
            cal = r.get("calibration", {})
            rows.append({"arm": arm, "seed": seed, "source": source, "auroc": e["auroc"], "auroc_lo": e["auroc_ci"][0], "auroc_hi": e["auroc_ci"][1],
                         "sensitivity": e["sensitivity"], "specificity": e["specificity"], "pr_auc": e["pr_auc"],
                         "ece": cal.get("ece"), "brier": cal.get("brier")})
collect(os.path.join(V2, "runs", "arm_D", "seed*", "results.json"), "D")
others = [f for f in glob.glob(os.path.join(INPUT, "**", "v2", "runs", "arm_*", "seed*", "results.json"), recursive=True)]
collect_files = others
for f in collect_files:
    collect(f, "others")
raw = pd.DataFrame(rows).drop_duplicates(["arm", "seed", "source"])
os.makedirs(os.path.join(WORK, "audit"), exist_ok=True)
raw.to_csv(os.path.join(WORK, "audit", "arm_D_comparison_per_seed.csv"), index=False)
out = []
for (source, arm), g in raw.groupby(["source", "arm"]):
    out.append({"source": source, "arm": arm, "seeds": len(g), "AUROC (95% CI)": f"{g.auroc.mean():.3f} ({g.auroc_lo.mean():.3f}-{g.auroc_hi.mean():.3f})",
                "PR-AUC": round(g.pr_auc.mean(), 3), "sensitivity": round(g.sensitivity.mean(), 3), "specificity": round(g.specificity.mean(), 3),
                "ECE": round(g.ece.mean(), 3), "Brier": round(g.brier.mean(), 3), "AUROC sd over seeds": round(g.auroc.std(), 3)})
table = pd.DataFrame(out)
table.to_csv(os.path.join(WORK, "audit", "arm_D_comparison.csv"), index=False)
for source, g in table.groupby("source"):
    print(f"\n=== {source}"); print(g.drop(columns="source").to_string(index=False))

## How to read it
* **D vs B on EyePACS:** if D is about as good as B, the gain comes from adding EyePACS, not from the balancing sampler. If D is clearly worse, balancing is doing real work.
* **D vs A on EyePACS:** if D beats A, a realistic merge helps on unseen patients of the added source; check APTOS to see whether that carries to a source the model never saw.
* **D vs C:** D keeps all EyePACS grades, so a specificity near 0.9 would show the arm C collapse (specificity 0.008) was caused by the label-skewed subset, not by merging itself.
* Whichever way it goes, report it: it is the most realistic merge in the study.